In [ ]:
# === BẢN 12B LƯỢNG TỬ + STREAM CỨU HỘ VÀO GOOGLE DRIVE ===
import os, sys, time, json, traceback, atexit, shutil
from pathlib import Path

print("Notebook 12B sạch: không có markdown dài, không hard-code token, có checkpoint chống mất phiên.")
print("Yêu cầu Colab: Runtime GPU T4, Colab Secret tên HF_TOKEN, rồi bấm Chạy tất cả.")

ACG_RUN_STAMP = time.strftime("%Y%m%d_%H%M%S")
ACG_DRIVE_DIR = None
ACG_LOCAL_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

class _ACGTee:
    def __init__(self, stream, file_path):
        self.stream = stream
        self.file_path = Path(file_path)
        self.file_path.parent.mkdir(parents=True, exist_ok=True)
        self.file = open(self.file_path, "a", encoding="utf-8", buffering=1)
        self._acg_tee = True
        self.encoding = getattr(stream, "encoding", "utf-8")
    def write(self, data):
        try:
            self.stream.write(data)
        except Exception:
            pass
        try:
            self.file.write(data)
            self.file.flush()
        except Exception:
            pass
    def flush(self):
        try:
            self.stream.flush()
        except Exception:
            pass
        try:
            self.file.flush()
        except Exception:
            pass
    def close(self):
        try:
            self.file.flush(); self.file.close()
        except Exception:
            pass
    def isatty(self):
        return False
    def __getattr__(self, name):
        return getattr(self.stream, name)

def _acg_mount_drive():
    global ACG_DRIVE_DIR
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        root = Path("/content/drive/MyDrive/acg_llm_fuzz_12b_runs")
        ACG_DRIVE_DIR = root / ACG_RUN_STAMP
        ACG_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        return ACG_DRIVE_DIR
    except Exception as e:
        ACG_DRIVE_DIR = None
        print(">> Không mount được Google Drive; vẫn lưu tạm trong runtime:", str(e)[:160])
        return None

def acg_artifact_path(name):
    return ACG_LOCAL_DIR / name

def acg_drive_path(name):
    return (Path(ACG_DRIVE_DIR) / name) if ACG_DRIVE_DIR else None

def acg_write_text(name, text):
    local = acg_artifact_path(name)
    local.parent.mkdir(parents=True, exist_ok=True)
    local.write_text(text, encoding="utf-8")
    dst = acg_drive_path(name)
    if dst:
        dst.parent.mkdir(parents=True, exist_ok=True)
        dst.write_text(text, encoding="utf-8")
    return str(local)

def acg_copy_to_drive(path, name=None):
    if not ACG_DRIVE_DIR:
        return None
    src = Path(path)
    if not src.exists():
        return None
    dst = Path(ACG_DRIVE_DIR) / (name or src.name)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return str(dst)

_drive_dir = _acg_mount_drive()
if _drive_dir:
    if not getattr(sys.stdout, "_acg_tee", False):
        sys.stdout = _ACGTee(sys.stdout, _drive_dir / "stdout_stream.log")
    if not getattr(sys.stderr, "_acg_tee", False):
        sys.stderr = _ACGTee(sys.stderr, _drive_dir / "stderr_stream.log")
    atexit.register(lambda: getattr(sys.stdout, "flush", lambda: None)())
    atexit.register(lambda: getattr(sys.stderr, "flush", lambda: None)())
    session = {
        "run_stamp": ACG_RUN_STAMP,
        "drive_dir": str(_drive_dir),
        "local_dir": str(ACG_LOCAL_DIR),
        "note": "stdout/stderr được tee trực tiếp vào Drive; checkpoint sẽ ghi song song vào Drive.",
    }
    acg_write_text("session_info.json", json.dumps(session, ensure_ascii=False, indent=2))
    print(">> Stream cứu hộ đang ghi vào Drive:", _drive_dir)
else:
    print(">> Chưa có Drive stream. Nếu cần cứu hộ khi mất kernel, chạy lại cell này và cho phép mount Drive.")

Notebook 12B sạch: không có markdown dài, không hard-code token, có checkpoint chống mất phiên.
Yêu cầu Colab: Runtime GPU T4, Colab Secret tên HF_TOKEN, rồi bấm Chạy tất cả.
Mounted at /content/drive
>> Stream cứu hộ đang ghi vào Drive: /content/drive/MyDrive/acg_llm_fuzz_12b_runs/20260610_011135


In [ ]:
# === 1. Môi trường chạy ===
import os, sys, time, json, math, random, subprocess, tempfile, select, re
from collections import Counter, defaultdict
from dataclasses import dataclass, field

# Dat truoc khi torch khoi tao CUDA de giam phan manh VRAM tren Colab.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

def _has(m):
    import importlib.util as u
    return u.find_spec(m) is not None

def _vparts(v):
    parts = [int(x) for x in re.findall(r"\d+", str(v))[:3]]
    return tuple(parts + [0] * (3 - len(parts)))

def _pkg_version(pkg):
    try:
        from importlib.metadata import version
        return version(pkg)
    except Exception:
        return "0"

def _pip_install(specs):
    if not specs:
        return
    print(">> Cài/cập nhật:", " ".join(specs))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", *specs], check=False)

def _ensure_core():
    miss = [p for p in ["numpy", "pandas", "matplotlib"] if not _has(p)]
    _pip_install(miss)

def _ensure_llm_stack():
    specs = []
    if _vparts(_pkg_version("transformers")) < _vparts("5.10.1"):
        specs.append("transformers>=5.10.1")
    if not _has("accelerate"):
        specs.append("accelerate")
    if not _has("bitsandbytes"):
        specs.append("bitsandbytes")
    for pkg, mod in [("huggingface_hub", "huggingface_hub"), ("sentencepiece", "sentencepiece"), ("protobuf", "google.protobuf"), ("safetensors", "safetensors")]:
        if not _has(mod):
            specs.append(pkg)
    _pip_install(specs)

_ensure_core()
HAVE_TORCH = _has("torch")
HAVE_TF = _has("tensorflow")
MOCK = os.environ.get("FUZZ_MOCK", "0") == "1" or not (HAVE_TORCH or HAVE_TF)
HAS_CUDA = False
if HAVE_TORCH and not MOCK:
    try:
        import torch
        HAS_CUDA = torch.cuda.is_available()
    except Exception:
        HAS_CUDA = False
print(">> MOCK:", MOCK, "| torch:", HAVE_TORCH, "| tf:", HAVE_TF, "| cuda:", HAS_CUDA)
import numpy as np

>> MOCK: False | torch: True | tf: True | cuda: True


In [ ]:
# === 2. Cấu hình 12B lượng tử ===
@dataclass
class Config:
    # Bản này khóa vào Gemma 4 12B 4-bit để ưu tiên chất lượng seed.
    model_id: str = "google/gemma-4-12B-it"
    model_candidates: tuple = ("google/gemma-4-12B-it",)
    model_preference: str = "12b"
    hf_token: str = field(default="", repr=False)  # lấy từ Colab Secret/env HF_TOKEN; không hard-code token vào notebook
    hf_token_env: str = "HF_TOKEN"
    use_llm: bool = True                    # False -> bo Gemma, chi fuzz random tren GPU
    time_budget_s: float = 1800.0       # 30 phút
    per_test_timeout: float = 4.5
    mem_mb: int = 4096
    cuda_frac: float = 0.18                 # worker torch duoc ~45% VRAM, phan con lai cho Gemma
    use_cuda: bool = True                   # bat oracle vi-sai CPU<->CUDA
    p_llm: float = 0.14                     # LLM dat huong; random/mutate giu throughput
    p_mutate: float = 0.68                  # tan dung corpus de dao sau cac mau da chay duoc
    corpus_cap: int = 1500
    min_budget: float = 1.0                 # tran thoi gian cho buoc tu toi gian (giay)
    gen_batch: int = 3                      # sinh theo batch roi dua vao hang doi
    prompt_max_tokens: int = 448
    max_new_tokens: int = 96
    llm_load_in_4bit: bool = True
    llm_use_double_quant: bool = True
    bnb_4bit_compute_dtype: str = "float16" # T4 hop voi fp16; A100/L4 co the doi bf16
    rng_seed: int = 12
    checkpoint_prefix: str = "acg_12b"
    checkpoint_every: int = 50

CFG = Config()
if MOCK:
    CFG.time_budget_s = 16.0
    CFG.per_test_timeout = 1.5
    CFG.use_cuda = True
else:
    CFG.use_cuda = HAS_CUDA
print(">> Cấu hình:", CFG)

>> Cấu hình: Config(model_id='google/gemma-4-12B-it', model_candidates=('google/gemma-4-12B-it',), model_preference='12b', hf_token_env='HF_TOKEN', use_llm=True, time_budget_s=1800.0, per_test_timeout=4.5, mem_mb=4096, cuda_frac=0.18, use_cuda=True, p_llm=0.14, p_mutate=0.68, corpus_cap=1500, min_budget=1.0, gen_batch=3, prompt_max_tokens=448, max_new_tokens=96, llm_load_in_4bit=True, llm_use_double_quant=True, bnb_4bit_compute_dtype='float16', rng_seed=12, checkpoint_prefix='acg_12b', checkpoint_every=50)


In [ ]:
# === 3. Ngữ pháp op-graph + bộ sinh ngẫu nhiên có hướng biên ===
DTYPES = ["float32", "float64", "int32", "int64"]
FILLS  = ["normal", "zeros", "nan", "inf", "big"]
OPS_REAL = ["matmul", "softmax", "reshape", "conv2d", "inv", "cumsum", "relu", "add", "exp"]
OPS_MOCK = OPS_REAL + ["slow"]

def _weighted(rng, pairs):
    vals, weights = zip(*pairs)
    return rng.choices(vals, weights=weights, k=1)[0]

def rand_shape(rng, allow_bad=True):
    rank = rng.choice([0, 1, 2, 3, 4, 5])
    out = []
    for _ in range(rank):
        c = rng.random()
        if allow_bad and c < 0.10:
            out.append(-rng.randint(1, 4))
        elif allow_bad and c < 0.16:
            out.append(0)
        elif allow_bad and c < 0.20:
            out.append(rng.choice([10**7, 10**8, 10**9]))
        else:
            out.append(rng.randint(1, 8))
    return out

def rand_tensor_step(rng, target=None, allow_bad=True):
    # Fuzzing hieu qua can phan lon mau chay duoc, nhung van chen edge-case am/0/khong lo.
    if target == "conv2d" and rng.random() < 0.80:
        shape = [rng.randint(1, 2), rng.randint(1, 4), rng.randint(1, 16), rng.randint(1, 16)]
    elif target in ("matmul", "inv") and rng.random() < 0.80:
        n = rng.randint(1, 8)
        shape = ([rng.randint(1, 3), n, n] if target == "matmul" and rng.random() < 0.35 else [n, n])
    elif target in ("softmax", "cumsum", "relu", "add", "exp") and rng.random() < 0.75:
        shape = [rng.randint(1, 6), rng.randint(1, 32)]
    else:
        shape = rand_shape(rng, allow_bad=allow_bad)
    if allow_bad and shape and rng.random() < 0.18:
        i = rng.randrange(len(shape))
        shape[i] = rng.choice([0, -rng.randint(1, 4), 10**8])
    return {
        "op": "tensor",
        "shape": shape,
        "dtype": _weighted(rng, [("float32", 5), ("float64", 2), ("int32", 1), ("int64", 1)]),
        "fill": _weighted(rng, [("normal", 7), ("zeros", 2), ("nan", 1), ("inf", 1), ("big", 1)]),
    }

def _reshape_target(rng):
    return rng.choice([[-1], [1, -1], [rng.randint(1, 8), -1], rand_shape(rng) or [-1]])

def random_program(rng, target=None, ops=None):
    ops = ops or OPS_REAL
    steps = [rand_tensor_step(rng, target)]
    n_ops = rng.choice([1, 1, 2, 3])
    for i in range(n_ops):
        o = target if (i == 0 and target in ops and rng.random() < 0.80) else rng.choice(ops)
        st = {"op": o}
        if o == "reshape": st["target"] = _reshape_target(rng)
        if o == "conv2d":  st["k"] = rng.choice([-1, 0, 1, 3])
        if o == "slow":    st["trigger"] = (rng.random() < 0.3)
        steps.append(st)
    return {"steps": steps}

def mutate_program(prog, rng, ops=None):
    ops = ops or OPS_REAL
    p = json.loads(json.dumps(prog))
    if not p.get("steps"):
        return random_program(rng, ops=ops)
    s = rng.choice(p["steps"])
    if s.get("op") == "tensor":
        k = rng.choice(["shape", "dtype", "fill"])
        s[k] = (rand_shape(rng) if k == "shape" else rng.choice(DTYPES if k == "dtype" else FILLS))
    else:
        if rng.random() < 0.55:
            s["op"] = rng.choice(ops)
        else:
            p["steps"].append({"op": rng.choice(ops)})
    return p

def prog_category(prog):
    last = prog["steps"][-1] if prog.get("steps") else {"op": "tensor"}
    t = prog["steps"][0] if prog.get("steps") else {}
    shp = t.get("shape", [])
    bad = any((isinstance(d, int) and (d < 0 or d == 0 or d >= 10**8)) for d in shp)
    return (last.get("op", "?"), min(len(shp), 5), t.get("fill", "normal"), "bad" if bad else "ok")

In [ ]:
# === 4. Bộ sinh: MockGenerator + RandomGenerator ===
class RandomGenerator:
    name="random"; is_llm=False
    def __init__(self, cfg, ops): self.ops=ops
    def generate(self, target, rng): return random_program(rng, target, self.ops)
    def generate_batch(self, target, rng, n): return [random_program(rng,target,self.ops) for _ in range(n)]
    def repair(self, prog, res, rng): return random_program(rng, None, self.ops)

class MockGenerator:
    name="mock-gen"; is_llm=True
    def __init__(self, cfg, ops): self.ops=ops
    def generate(self, target, rng):
        if rng.random()<0.08:   # thinh thoang sinh op la -> kich hoat tu-sua
            return {"steps":[{"op":"tensor","shape":rand_shape(rng),"dtype":rng.choice(DTYPES),
                              "fill":rng.choice(FILLS)},{"op":"frobnicate"}]}
        return random_program(rng, target, self.ops)
    def generate_batch(self, target, rng, n): return [self.generate(target,rng) for _ in range(n)]
    def repair(self, prog, res, rng):
        return random_program(rng, None, [o for o in self.ops if o!="frobnicate"])

In [ ]:
# === 5. GemmaGenerator 12B: 4-bit, sinh theo batch queue ===
class GemmaGenerator:
    name = "gemma"; is_llm = True
    SYSTEM_PROMPT = (
        "You generate compact JSON op-graphs for deep-learning API fuzzing. "
        "Return exactly one JSON object and no prose. The schema is "
        "{\"steps\":[{\"op\":\"tensor\",\"shape\":[...],\"dtype\":\"float32\",\"fill\":\"normal\"}, ...]}. "
        "Valid ops after tensor: matmul, softmax, reshape, conv2d, inv, cumsum, relu, add, exp. "
        "Use edge cases such as zero/negative/large dimensions, nan, inf, big values, unusual dtypes, "
        "but keep the graph small and executable."
    )
    def __init__(self, cfg, ops):
        import torch
        from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
        try:
            from transformers import AutoModelForMultimodalLM
        except Exception:
            AutoModelForMultimodalLM = None
        try:
            from transformers import AutoModelForImageTextToText
        except Exception:
            AutoModelForImageTextToText = None

        self.cfg = cfg; self.ops = ops; self.torch = torch; self.model_id = cfg.model_id; self.queue = []
        token_kw = {"token": cfg.hf_token} if cfg.hf_token else {}
        self.processor = AutoProcessor.from_pretrained(cfg.model_id, **token_kw)
        self.tok = getattr(self.processor, "tokenizer", self.processor)
        if getattr(self.tok, "pad_token", None) is None and getattr(self.tok, "eos_token", None) is not None:
            self.tok.pad_token = self.tok.eos_token
        if hasattr(self.tok, "padding_side"):
            self.tok.padding_side = "left"

        qconf = None
        if cfg.llm_load_in_4bit:
            compute_dtype = getattr(torch, cfg.bnb_4bit_compute_dtype, torch.float16)
            qconf = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=compute_dtype,
                bnb_4bit_use_double_quant=cfg.llm_use_double_quant,
            )
        load_kwargs = dict(device_map="auto", dtype="auto", low_cpu_mem_usage=True, **token_kw)
        if qconf is not None:
            load_kwargs["quantization_config"] = qconf

        candidates = [AutoModelForMultimodalLM, AutoModelForImageTextToText, AutoModelForCausalLM]
        last_err = None
        for cls in [c for c in candidates if c is not None]:
            try:
                self.model = cls.from_pretrained(cfg.model_id, **load_kwargs)
                self.model_class = cls.__name__
                break
            except Exception as e:
                last_err = e
        else:
            raise last_err
        self.model.eval()
        try:
            print(">> Gemma VRAM footprint: %.2f GB" % (self.model.get_memory_footprint() / 1024**3))
        except Exception:
            pass

    def _device(self):
        try:
            return self.model.device
        except Exception:
            return next(self.model.parameters()).device

    def _prompt(self, target):
        target = target or random.choice(self.ops)
        ex = '{"steps":[{"op":"tensor","shape":[3,3],"dtype":"float64","fill":"big"},{"op":"%s"}]}' % target
        return (
            "Target op: `%s`. Generate one high-value fuzzing program. "
            "Prefer 1-4 steps, include at least one tensor step first, and output JSON only. "
            "Allowed ops: %s. Example shape/op format: %s" % (target, ",".join(self.ops), ex)
        )

    def _chat_text(self, prompt):
        messages = [{"role": "system", "content": self.SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
        tmpl = getattr(self.tok, "apply_chat_template", None) or getattr(self.processor, "apply_chat_template", None)
        if tmpl:
            try:
                return tmpl(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
            except TypeError:
                try:
                    return tmpl(messages, tokenize=False, add_generation_prompt=True)
                except Exception:
                    pass
            except Exception:
                pass
        return self.SYSTEM_PROMPT + "\n" + prompt + "\nJSON:"

    def _tokenize(self, texts):
        enc = self.tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=self.cfg.prompt_max_tokens)
        return enc.to(self._device())

    def generate_batch(self, target, rng, n):
        torch = self.torch
        try:
            prompts = [self._chat_text(self._prompt(target)) for _ in range(n)]
            enc = self._tokenize(prompts)
            gen_kwargs = dict(
                max_new_tokens=self.cfg.max_new_tokens,
                do_sample=True,
                temperature=1.0,
                top_p=0.95,
                top_k=64,
            )
            if getattr(self.tok, "pad_token_id", None) is not None:
                gen_kwargs["pad_token_id"] = self.tok.pad_token_id
            with torch.inference_mode():
                out = self.model.generate(**enc, **gen_kwargs)
            input_len = enc["input_ids"].shape[1]
            texts = self.tok.batch_decode(out[:, input_len:], skip_special_tokens=True)
            return [self._parse(t, target, rng) for t in texts]
        except Exception as e:
            if not getattr(self, "_warned", False):
                print(">> Gemma sinh loi (%s); tam dung random cho batch nay." % str(e)[:120])
                self._warned = True
            return [random_program(rng, target, self.ops) for _ in range(n)]

    def generate(self, target, rng):
        if not self.queue:
            self.queue.extend(self.generate_batch(target, rng, max(1, self.cfg.gen_batch)))
        return self.queue.pop(0)

    def _shape(self, val):
        if not isinstance(val, list):
            return []
        out = []
        for d in val[:5]:
            try:
                d = int(d)
            except Exception:
                d = 1
            if abs(d) > 10**9:
                d = 10**9 if d > 0 else -10**9
            out.append(d)
        return out

    def _normalize(self, p, target, rng):
        if not isinstance(p, dict) or not isinstance(p.get("steps"), list):
            return random_program(rng, target, self.ops)
        steps = []
        for st in p["steps"][:5]:
            if not isinstance(st, dict):
                continue
            op = st.get("op")
            if op == "tensor":
                steps.append({
                    "op": "tensor",
                    "shape": self._shape(st.get("shape", [])),
                    "dtype": st.get("dtype") if st.get("dtype") in DTYPES else "float32",
                    "fill": st.get("fill") if st.get("fill") in FILLS else "normal",
                })
            elif op in self.ops:
                clean = {"op": op}
                if op == "reshape": clean["target"] = self._shape(st.get("target", [-1])) or [-1]
                if op == "conv2d":
                    try: clean["k"] = int(st.get("k", 1))
                    except Exception: clean["k"] = 1
                steps.append(clean)
        if not steps or steps[0].get("op") != "tensor":
            steps.insert(0, rand_tensor_step(rng, target))
        if len(steps) == 1:
            steps.append({"op": target if target in self.ops else rng.choice(self.ops)})
        return {"steps": steps}

    def _parse(self, text, target, rng):
        text = text.replace("```json", "```").replace("```", " ").strip()
        dec = json.JSONDecoder()
        for m in re.finditer(r"\{", text):
            try:
                obj, _ = dec.raw_decode(text[m.start():])
                return self._normalize(obj, target, rng)
            except Exception:
                continue
        return random_program(rng, target, self.ops)

    def repair(self, prog, res, rng):
        return self.generate(rng.choice(self.ops), rng)

In [ ]:
# === 6. Mã worker thường trú, chạy trong tiến trình con ===
WORKER_SRC = r"""
import sys, json, os
FW = sys.argv[1] if len(sys.argv)>1 else "mock"
MEM = int(sys.argv[2]) if len(sys.argv)>2 else 4096
CFRAC = float(sys.argv[3]) if len(sys.argv)>3 else 0.35
def _limits():
    try:
        import resource
        resource.setrlimit(resource.RLIMIT_CORE,(0,0))
        if FW=="mock":
            b=MEM*1024*1024; resource.setrlimit(resource.RLIMIT_AS,(b,b))
    except Exception: pass
_limits()
_TORCH=None
def _torch():
    global _TORCH
    if _TORCH is None:
        import torch; _TORCH=torch
        try:
            if torch.cuda.is_available(): torch.cuda.set_per_process_memory_fraction(CFRAC,0)
        except Exception: pass
    return _TORCH

def mock_exec(prog, device):
    import ctypes, hashlib
    val=None; nonfin=False
    for st in prog.get("steps",[]):
        op=st.get("op")
        if op=="tensor":
            val={"shape":st.get("shape",[]),"fill":st.get("fill","normal"),"dtype":st.get("dtype","float32")}
            if val["fill"] in ("inf","nan"): nonfin=True
            continue
        if val is None: return {"status":"invalid","detail":"no tensor"}
        sh=val["shape"]
        if op=="conv2d":
            if int(st.get("k",1))<=0 or any(d<0 for d in sh): os.abort()
        elif op=="matmul":
            if any(d>50000000 for d in sh): ctypes.string_at(0)
            if len(sh)<2: return {"status":"exception","etype":"ValueError"}
        elif op=="inv":
            if device=="cuda": nonfin=True
        elif op=="reshape":
            numel=1
            for d in sh: numel*= d if d>0 else 0
            prod=1
            for d in st.get("target",[1]):
                if d>0: prod*=d
            if prod and numel%prod: return {"status":"exception","etype":"ValueError"}
        elif op in ("exp","softmax"):
            if val["fill"]=="inf": nonfin=True
        elif op=="slow":
            if st.get("trigger"):
                while True: pass
        elif op=="frobnicate":
            return {"status":"invalid","detail":"unknown op"}
    if nonfin: return {"status":"silent_nonfinite","fp":"nonfinite"}
    base=hashlib.blake2b(json.dumps(prog,sort_keys=True).encode(),digest_size=4).hexdigest()
    fp=base+('' if device in ("cpu","cuda") else "_dev")
    return {"status":"ok","fp":fp}

def _numel(shape):
    n=1
    for d in shape: n*=int(d)
    return n

def _deterministic_tensor(torch, shape, dt):
    n=_numel(shape)
    if dt.is_floating_point:
        base=torch.arange(n,dtype=torch.float32).reshape(shape)
        return (((base % 17) - 8) / 8).to(dtype=dt)
    return (torch.arange(n,dtype=torch.int64).reshape(shape) % 5).to(dtype=dt)

def _mk(torch, st, device):
    dt=getattr(torch, st.get("dtype","float32")); shape=st.get("shape",[]); fill=st.get("fill","normal")
    if fill=="nan": v=torch.full(shape,float("nan"),dtype=dt)
    elif fill=="inf": v=torch.full(shape,float("inf"),dtype=dt)
    elif fill=="big": v=torch.full(shape,1e30,dtype=dt) if dt.is_floating_point else torch.full(shape,2**30,dtype=dt)
    elif fill=="zeros": v=torch.zeros(shape,dtype=dt)
    else: v=_deterministic_tensor(torch, shape, dt)
    return v.to("cuda") if device=="cuda" else v

def torch_exec(prog, device):
    torch=_torch(); F=torch.nn.functional
    if device=="cuda" and not torch.cuda.is_available(): return {"status":"invalid","detail":"no cuda"}
    val=None
    for st in prog.get("steps",[]):
        op=st.get("op")
        try:
            if op=="tensor": val=_mk(torch, st, device); continue
            if val is None: return {"status":"invalid","detail":"no tensor"}
            if op=="matmul": val=torch.matmul(val,val)
            elif op=="softmax": val=F.softmax(val,dim=-1)
            elif op=="reshape": val=val.reshape(st.get("target",[-1]))
            elif op=="conv2d":
                base=val.float() if not val.is_floating_point() else val
                in_ch=base.shape[1] if base.dim()>1 else 1
                w=torch.ones((1,in_ch,1,1),device=base.device,dtype=base.dtype)
                val=F.conv2d(base, w)
            elif op=="inv": val=torch.linalg.inv(val)
            elif op=="cumsum": val=torch.cumsum(val,dim=-1)
            elif op=="relu": val=F.relu(val)
            elif op=="add": val=val+val
            elif op=="exp": val=torch.exp(val)
            else: return {"status":"invalid","detail":"unknown op "+str(op)}
        except RuntimeError as e:
            m=str(e).lower()
            if "out of memory" in m or "alloc" in m:
                try: torch.cuda.empty_cache()
                except Exception: pass
                return {"status":"resource","etype":"OOM","msg":str(e)[:100]}
            return {"status":"exception","etype":"RuntimeError","msg":str(e)[:100]}
        except Exception as e:
            return {"status":"exception","etype":type(e).__name__,"msg":str(e)[:100]}
    try:
        if hasattr(val,"is_floating_point") and val.is_floating_point():
            if not bool(torch.isfinite(val).all()): return {"status":"silent_nonfinite","fp":"nonfinite"}
            fp=round(float(val.float().abs().sum().item())%1e6,2)
        else: fp="int"
    except Exception: fp="na"
    try: torch.cuda.empty_cache()
    except Exception: pass
    return {"status":"ok","fp":fp}

def execute(task):
    prog=task["program"]; device=task.get("device","cpu")
    if FW=="mock": return mock_exec(prog, device)
    if FW=="torch": return torch_exec(prog, device)
    return {"status":"invalid","detail":"fw "+FW}

for line in sys.stdin:
    line=line.strip()
    if not line: continue
    try: task=json.loads(line)
    except Exception:
        sys.stdout.write(json.dumps({"status":"exception","etype":"BadTask"})+"\n"); sys.stdout.flush(); continue
    r=execute(task)
    sys.stdout.write(json.dumps(r)+"\n"); sys.stdout.flush()
"""
WORKER_PATH=os.path.join(tempfile.gettempdir(),"_acg_worker_12b.py")
with open(WORKER_PATH,"w",encoding="utf-8") as f: f.write(WORKER_SRC)
print(">> Worker:", WORKER_PATH)

>> Worker: /tmp/_acg_worker_12b.py


In [ ]:
# === 7. Executor thường trú cách ly + vi-sai CPU/CUDA + tối giản ===
class PersistentExecutor:
    SIGNALS={11:"SIGSEGV",6:"SIGABRT",8:"SIGFPE",4:"SIGILL",7:"SIGBUS",9:"SIGKILL"}
    def __init__(self, framework, cfg):
        self.fw=framework; self.cfg=cfg; self.p=None; self._start()
    def _start(self):
        self.p=subprocess.Popen([sys.executable, WORKER_PATH, self.fw, str(self.cfg.mem_mb), str(self.cfg.cuda_frac)],
            stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True, bufsize=1)
    def _restart(self):
        try: self.p.kill()
        except Exception: pass
        self._start()
    def run_one(self, prog, device, cfg):
        try:
            self.p.stdin.write(json.dumps({"program":prog,"device":device})+"\n"); self.p.stdin.flush()
        except (BrokenPipeError, OSError):
            self._restart(); return {"status":"crash","signal":"PIPE"}
        t0=time.time()
        while True:
            rem=cfg.per_test_timeout-(time.time()-t0)
            if rem<=0: self._restart(); return {"status":"hang"}
            r,_,_=select.select([self.p.stdout],[],[],rem)
            if not r: self._restart(); return {"status":"hang"}
            out=self.p.stdout.readline()
            if out=="":
                try: self.p.wait(timeout=1)
                except Exception: pass
                code=self.p.returncode; self._restart()
                if code is not None and code<0:
                    return {"status":"crash","signal":self.SIGNALS.get(-code,"SIG%d"%(-code))}
                return {"status":"crash","signal":"EXIT%s"%code}
            out=out.strip()
            if not out: continue
            try: return json.loads(out)
            except Exception: continue
    def run_devdiff(self, prog, cfg):
        rc=self.run_one(prog,"cpu",cfg)
        if not cfg.use_cuda: rc["used_cuda"]=False; return rc
        rg=self.run_one(prog,"cuda",cfg)
        bad=("crash","hang"); sc=rc["status"]; sg=rg["status"]; o={"used_cuda":True}
        if sg in bad and sc not in bad: o.update({"status":sg,"signal":rg.get("signal",""),"detail":"cuda-only"}); return o
        if sc in bad and sg not in bad: o.update({"status":sc,"signal":rc.get("signal",""),"detail":"cpu-only"}); return o
        if sc in bad and sg in bad:
            o.update({"status":"hang" if (sc=="hang" and sg=="hang") else "crash",
                      "signal":(rg.get("signal") or rc.get("signal") or ""),"detail":"both"}); return o
        if "invalid" in (sc,sg): o["status"]="invalid"; return o
        if sc=="silent_nonfinite" or sg=="silent_nonfinite":
            o.update({"status":"silent_nonfinite","detail":("cuda" if sg=="silent_nonfinite" else "cpu")}); return o
        fc=rc.get("fp"); fg=rg.get("fp")
        if fc!=fg: o.update({"status":"device_divergence","detail":"cpu=%s cuda=%s"%(fc,fg)}); return o
        o.update({"status":"ok","fp":fc}); return o
    def minimize(self, prog, res, cfg):
        tgt=res["status"]; steps=prog.get("steps",[])[:]; changed=True; guard=0; t0=time.time()
        while changed and len(steps)>1 and guard<12 and time.time()-t0<cfg.min_budget:
            changed=False; guard+=1
            for i in range(len(steps)-1,0,-1):
                trial={"steps":steps[:i]+steps[i+1:]}
                if self.run_devdiff(trial,cfg)["status"]==tgt:
                    steps=trial["steps"]; changed=True; break
        return {"steps":steps}

In [ ]:
# === 8. Vòng phản hồi động: độ hiếm x độ dễ vỡ ===
class Feedback:
    def __init__(self): self.tries=Counter(); self.frag=defaultdict(float); self.cov=Counter()
    def pick(self, rng, ops):
        ws=[(1.0/(1+self.tries[o]))*(1.0+self.frag[o]) for o in ops]
        return rng.choices(ops, weights=ws, k=1)[0]
    def observe(self, op, status):
        self.tries[op]+=1
        if status in ("crash","hang","silent_nonfinite","device_divergence","resource"): self.frag[op]+=1.0
        elif status=="ok": self.frag[op]=max(0.0, self.frag[op]*0.97)
    def cover(self, cat): self.cov[cat]+=1

In [ ]:
# === 9. Vòng fuzz chính ===
def fuzz(gen, ex, cfg, ops):
    rng=random.Random(cfg.rng_seed); fb=Feedback()
    corpus=[]; crashes=[]; seen=set(); curve=[]; stats=Counter()
    total=0; valid=0; gpu=0; t0=time.time(); it=0
    print("="*72)
    print(" BAT DAU FUZZ | generator=%s | oracle=%s | ngan sach=%.0fs" % (gen.name, ("vi-sai CPU+CUDA" if cfg.use_cuda else "CPU-only"), cfg.time_budget_s))
    print(" Moi BUG duy nhat in 1 dong; tien do in moi 10 test.")
    print("="*72)
    while time.time()-t0<cfg.time_budget_s:
        it+=1
        target=fb.pick(rng, ops); roll=rng.random()
        if roll<cfg.p_llm: prog=gen.generate(target,rng); src=gen.name
        elif corpus and roll<cfg.p_llm+cfg.p_mutate: prog=mutate_program(rng.choice(corpus),rng,ops); src="mutate"
        else: prog=random_program(rng,target,ops); src="random"
        res=ex.run_devdiff(prog,cfg); total+=1
        if res.get("used_cuda"): gpu+=1
        if res["status"]=="invalid":
            prog=gen.repair(prog,res,rng); res=ex.run_devdiff(prog,cfg); stats["repaired"]+=1
        if res["status"]!="invalid": valid+=1
        stats[res["status"]]+=1
        cat=prog_category(prog); fb.cover(cat); lastop=cat[0]; fb.observe(lastop,res["status"])
        if res["status"] in ("ok","silent_nonfinite") and src!="random":
            corpus.append(prog)
            if len(corpus)>cfg.corpus_cap: corpus.pop(0)
        if res["status"] in ("crash","hang","device_divergence"):
            key=(lastop,res["status"],res.get("signal",""),str(res.get("detail",""))[:48])
            if key not in seen:
                seen.add(key); mn=ex.minimize(prog,res,cfg)
                attop=mn["steps"][-1]["op"] if mn.get("steps") else lastop
                crashes.append({"iter":it,"op":attop,"status":res["status"],"signal":res.get("signal",""),
                    "detail":str(res.get("detail","")),"src":src,
                    "steps_full":len(prog.get("steps",[])),"steps_min":len(mn["steps"]),"program":mn})
                print("  >> BUG #%-2d  %-14s %-10s %-9s | rut gon %d->%d buoc | nguon=%s" % (len(seen), attop, res["status"], (res.get("signal","") or "-"), len(prog.get("steps",[])), len(mn["steps"]), src))
        curve.append((total,len(seen),valid))
        if it%10==0:
            elapsed=max(1e-9,time.time()-t0)
            msg="[%4d test] hop_le %5.1f%% | BUG %2d | gpu %4d | %.2f test/s | crash %d hang %d lech %d nan %d ok %d | %4.1fs" % (total,100*valid/max(1,total),len(seen),gpu,total/elapsed,stats["crash"],stats["hang"],stats["device_divergence"],stats["silent_nonfinite"],stats["ok"],elapsed)
            print(msg)
    print("\n"+"="*72)
    print(" KET THUC: %d test | hop le %.1f%% | %d BUG DUY NHAT | corpus %d | gpu %d | %d nhom-input" % (total,100*valid/max(1,total),len(seen),len(corpus),gpu,len(fb.cov)))
    print(" Phan bo trang thai:", {k: v for k, v in sorted(stats.items())})
    print(" (chi tiet BUG + reproducer da toi gian o cell bao cao ben duoi)")
    print("="*72)
    return {"crashes":crashes,"curve":curve,"stats":dict(stats),"total":total,
            "valid":valid,"seen":len(seen),"gpu_runs":gpu,"cov":len(fb.cov)}

In [ ]:
# === Ghi đè vòng fuzz: checkpoint sống cho bản 12B ===
def _atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def _save_checkpoint(cfg, crashes, curve, stats, total, valid, seen, gpu, cov):
    prefix = getattr(cfg, "checkpoint_prefix", "acg_12b")
    crash_text = "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in crashes)
    meta = {
        "total": total, "valid": valid, "seen": len(seen), "gpu_runs": gpu,
        "cov": cov, "stats": dict(stats), "curve_tail": curve[-50:], "time": time.time(),
        "drive_dir": str(globals().get("ACG_DRIVE_DIR") or ""),
    }
    progress_text = json.dumps(meta, ensure_ascii=False, indent=2)

    local_crashes = acg_artifact_path(prefix + "_crashes_live.jsonl") if "acg_artifact_path" in globals() else Path(prefix + "_crashes_live.jsonl")
    local_progress = acg_artifact_path(prefix + "_progress.json") if "acg_artifact_path" in globals() else Path(prefix + "_progress.json")
    _atomic_write_text(local_crashes, crash_text)
    _atomic_write_text(local_progress, progress_text)

    if globals().get("ACG_DRIVE_DIR"):
        drive_crashes = Path(ACG_DRIVE_DIR) / (prefix + "_crashes_live.jsonl")
        drive_progress = Path(ACG_DRIVE_DIR) / (prefix + "_progress.json")
        _atomic_write_text(drive_crashes, crash_text)
        _atomic_write_text(drive_progress, progress_text)


def fuzz(gen, ex, cfg, ops):
    rng=random.Random(cfg.rng_seed); fb=Feedback()
    corpus=[]; crashes=[]; seen=set(); curve=[]; stats=Counter()
    total=0; valid=0; gpu=0; t0=time.time(); it=0
    print("="*72)
    print(" BẮT ĐẦU FUZZ 12B | generator=%s | oracle=%s | ngân sách=%.0fs" % (gen.name, ("vi-sai CPU+CUDA" if cfg.use_cuda else "CPU-only"), cfg.time_budget_s))
    print(" Mỗi BUG duy nhất in 1 dòng; checkpoint ghi mỗi %d test." % getattr(cfg, "checkpoint_every", 50))
    print("="*72)
    while time.time()-t0<cfg.time_budget_s:
        it+=1
        target=fb.pick(rng, ops); roll=rng.random()
        if roll<cfg.p_llm: prog=gen.generate(target,rng); src=gen.name
        elif corpus and roll<cfg.p_llm+cfg.p_mutate: prog=mutate_program(rng.choice(corpus),rng,ops); src="mutate"
        else: prog=random_program(rng,target,ops); src="random"
        res=ex.run_devdiff(prog,cfg); total+=1
        if res.get("used_cuda"): gpu+=1
        if res["status"]=="invalid":
            prog=gen.repair(prog,res,rng); res=ex.run_devdiff(prog,cfg); stats["repaired"]+=1
        if res["status"]!="invalid": valid+=1
        stats[res["status"]]+=1
        cat=prog_category(prog); fb.cover(cat); lastop=cat[0]; fb.observe(lastop,res["status"])
        if res["status"] in ("ok","silent_nonfinite") and src!="random":
            corpus.append(prog)
            if len(corpus)>cfg.corpus_cap: corpus.pop(0)
        if res["status"] in ("crash","hang","device_divergence"):
            key=(lastop,res["status"],res.get("signal",""),str(res.get("detail",""))[:48])
            if key not in seen:
                seen.add(key); mn=ex.minimize(prog,res,cfg)
                attop=mn["steps"][-1]["op"] if mn.get("steps") else lastop
                crashes.append({"iter":it,"op":attop,"status":res["status"],"signal":res.get("signal",""),
                    "detail":str(res.get("detail","")),"src":src,
                    "steps_full":len(prog.get("steps",[])),"steps_min":len(mn["steps"]),"program":mn})
                print("  >> BUG #%-2d  %-14s %-10s %-9s | rút gọn %d->%d bước | nguồn=%s" % (len(seen), attop, res["status"], (res.get("signal","") or "-"), len(prog.get("steps",[])), len(mn["steps"]), src))
                _save_checkpoint(cfg, crashes, curve, stats, total, valid, seen, gpu, len(fb.cov))
        curve.append((total,len(seen),valid))
        if it%10==0:
            elapsed=max(1e-9,time.time()-t0)
            msg="[%4d test] hợp_lệ %5.1f%% | BUG %2d | gpu %4d | %.2f test/s | crash %d hang %d lệch %d nan %d ok %d | %4.1fs" % (total,100*valid/max(1,total),len(seen),gpu,total/elapsed,stats["crash"],stats["hang"],stats["device_divergence"],stats["silent_nonfinite"],stats["ok"],elapsed)
            print(msg)
        if it % max(1, getattr(cfg, "checkpoint_every", 50)) == 0:
            _save_checkpoint(cfg, crashes, curve, stats, total, valid, seen, gpu, len(fb.cov))
    _save_checkpoint(cfg, crashes, curve, stats, total, valid, seen, gpu, len(fb.cov))
    print("\n"+"="*72)
    print(" KẾT THÚC: %d test | hợp lệ %.1f%% | %d BUG DUY NHẤT | corpus %d | gpu %d | %d nhóm-input" % (total,100*valid/max(1,total),len(seen),len(corpus),gpu,len(fb.cov)))
    print(" Phân bố trạng thái:", {k: v for k, v in sorted(stats.items())})
    ck_name = getattr(cfg, "checkpoint_prefix", "acg_12b") + "_crashes_live.jsonl"
    print(" Da luu checkpoint:", acg_artifact_path(ck_name) if "acg_artifact_path" in globals() else ck_name)
    if globals().get("ACG_DRIVE_DIR"):
        print(" Ban cuu ho tren Drive:", Path(ACG_DRIVE_DIR) / ck_name)
    print("="*72)
    return {"crashes":crashes,"curve":curve,"stats":dict(stats),"total":total,
            "valid":valid,"seen":len(seen),"gpu_runs":gpu,"cov":len(fb.cov)}

In [ ]:
# === 10. Lắp ráp + chạy Gemma 4 12B 4-bit ===
OPS = OPS_MOCK if MOCK else OPS_REAL

def _get_hf_token(cfg):
    tok = cfg.hf_token or os.environ.get(cfg.hf_token_env, "")
    if not tok:
        try:
            from google.colab import userdata
            tok = userdata.get(cfg.hf_token_env) or ""
            if tok:
                os.environ[cfg.hf_token_env] = tok
        except Exception:
            pass
    return tok

def _candidate_models(cfg):
    # B?n n?y ch? ch?y Gemma 4 12B; n?u 12B kh?ng n?p ???c th? fallback random ?? v?n fuzz GPU.
    return [cfg.model_id or "google/gemma-4-12B-it"]

if not MOCK and HAS_CUDA and CFG.use_llm:
    _ensure_llm_stack()

_tok = _get_hf_token(CFG)
if _tok:
    CFG.hf_token = _tok
    try:
        from huggingface_hub import login
        login(token=_tok, add_to_git_credential=False)
        print(">> Đã đăng nhập Hugging Face bằng secret/env HF_TOKEN.")
    except Exception as e:
        print(">> Đăng nhập HF lỗi:", str(e)[:120])
else:
    print(">> Chưa có HF_TOKEN; nếu gặp 401/403, thêm Colab secret tên HF_TOKEN rồi chạy lại runtime.")

def build_generator():
    if MOCK:
        return MockGenerator(CFG, OPS)
    if HAS_CUDA and CFG.use_llm:
        last = None
        for mid in _candidate_models(CFG):
            try:
                CFG.model_id = mid
                print(">> Nạp Gemma 12B lượng tử:", mid)
                g = GemmaGenerator(CFG, OPS)
                print(">> Đã nạp Gemma:", mid, "| class:", getattr(g, "model_class", "?"))
                return g
            except Exception as e:
                last = e
                line = (str(e).splitlines() or [""])[0]
                print(">> KHÔNG nạp được '%s': %s" % (mid, line[:140]))
                try:
                    import torch; torch.cuda.empty_cache()
                except Exception:
                    pass
        print(">> -> Fallback RandomGenerator (van fuzz tren GPU, chi thieu phan LLM dan duong).")
        if last:
            print(">> Lý do cuối:", str(last)[:180])
        return RandomGenerator(CFG, OPS)
    return RandomGenerator(CFG, OPS)

GEN = build_generator()
EX  = PersistentExecutor("mock" if MOCK else "torch", CFG)
print(">> Generator:", GEN.name, "| model:", getattr(GEN, "model_id", "-"), "| Executor fw:", EX.fw, "| #ops:", len(OPS))
RESULT = fuzz(GEN, EX, CFG, OPS)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


>> Đã đăng nhập Hugging Face bằng secret/env HF_TOKEN.
>> Nạp Gemma 12B lượng tử: google/gemma-4-12B-it


model.safetensors:   0%|          | 0.00/23.9G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

>> Gemma VRAM footprint: 6.99 GB
>> Đã nạp Gemma: google/gemma-4-12B-it | class: AutoModelForMultimodalLM
>> Generator: gemma | model: google/gemma-4-12B-it | Executor fw: torch | #ops: 9
 BẮT ĐẦU FUZZ 12B | generator=gemma | oracle=vi-sai CPU+CUDA | ngân sách=1800s
 Mỗi BUG duy nhất in 1 dòng; checkpoint ghi mỗi 50 test.
  >> BUG #1   tensor         hang       -         | rút gọn 2->1 bước | nguồn=random
[  10 test] hợp_lệ 100.0% | BUG  1 | gpu   10 | 0.12 test/s | crash 0 hang 1 lệch 0 nan 0 ok 9 | 84.9s
[  20 test] hợp_lệ 100.0% | BUG  1 | gpu   20 | 0.14 test/s | crash 0 hang 1 lệch 0 nan 0 ok 19 | 146.3s
[  30 test] hợp_lệ 100.0% | BUG  1 | gpu   30 | 0.20 test/s | crash 0 hang 1 lệch 0 nan 2 ok 27 | 146.6s
  >> BUG #2   matmul         device_divergence -         | rút gọn 2->2 bước | nguồn=gemma
  >> BUG #3   exp            hang       -         | rút gọn 4->3 bước | nguồn=random
[  40 test] hợp_lệ 100.0% | BUG  3 | gpu   40 | 0.18 test/s | crash 0 hang 2 lệch 1 nan 2 ok 35 | 228.

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import os, json

print("ACG_DRIVE_DIR =", globals().get("ACG_DRIVE_DIR"))

if globals().get("ACG_DRIVE_DIR"):
    d = Path(ACG_DRIVE_DIR)
    print("exists:", d.exists())
    for p in sorted(d.glob("*")):
        print(p.name, p.stat().st_size, "bytes")
else:
    print("Chưa thấy biến ACG_DRIVE_DIR. Có thể cell đầu chưa chạy hoặc runtime đã reset.")
    print("Tìm thủ công trong Drive: MyDrive/acg_llm_fuzz_12b_runs/")

ACG_DRIVE_DIR = /content/drive/MyDrive/acg_llm_fuzz_12b_runs/20260610_011135
exists: True
acg_12b_crashes_live.jsonl 4104 bytes
acg_12b_progress.json 2516 bytes
session_info.json 250 bytes
stderr_stream.log 992 bytes
stdout_stream.log 10029 bytes


In [ ]:
# === 12B FAST MODE: dùng 12B làm seed oracle, không gọi quá thường xuyên ===
try:
    EX.p.kill()
except Exception:
    pass

CFG.time_budget_s = 100.0
CFG.per_test_timeout = 6.0
CFG.cuda_frac = 0.22

CFG.p_llm = 0.04        # 12B chỉ thỉnh thoảng sinh seed xịn
CFG.p_mutate = 0.76     # đào sâu từ corpus
CFG.gen_batch = 2
CFG.max_new_tokens = 72
CFG.prompt_max_tokens = 384
CFG.checkpoint_every = 25

if hasattr(GEN, "queue"):
    GEN.queue.clear()

EX = PersistentExecutor("torch", CFG)
print(">> 12B FAST MODE:", CFG)

RESULT_FAST12B = fuzz(GEN, EX, CFG, OPS)
RESULT = RESULT_FAST12B

>> 12B FAST MODE: Config(model_id='google/gemma-4-12B-it', model_candidates=('google/gemma-4-12B-it',), model_preference='12b', hf_token_env='HF_TOKEN', use_llm=True, time_budget_s=100.0, per_test_timeout=6.0, mem_mb=4096, cuda_frac=0.22, use_cuda=True, p_llm=0.04, p_mutate=0.76, corpus_cap=1500, min_budget=1.0, gen_batch=2, prompt_max_tokens=384, max_new_tokens=72, llm_load_in_4bit=True, llm_use_double_quant=True, bnb_4bit_compute_dtype='float16', rng_seed=12, checkpoint_prefix='acg_12b', checkpoint_every=25)
 BẮT ĐẦU FUZZ 12B | generator=gemma | oracle=vi-sai CPU+CUDA | ngân sách=100s
 Mỗi BUG duy nhất in 1 dòng; checkpoint ghi mỗi 25 test.
  >> BUG #1   tensor         hang       -         | rút gọn 2->1 bước | nguồn=random
[  10 test] hợp_lệ 100.0% | BUG  1 | gpu   10 | 0.30 test/s | crash 0 hang 1 lệch 0 nan 0 ok 9 | 33.3s
[  20 test] hợp_lệ 100.0% | BUG  1 | gpu   20 | 0.26 test/s | crash 0 hang 1 lệch 0 nan 0 ok 19 | 77.4s
[  30 test] hợp_lệ 100.0% | BUG  1 | gpu   30 | 0.39 test

In [ ]:
# === 11. Báo cáo + lưu kết quả 12B ===
%matplotlib inline
import pandas as pd
import matplotlib.pyplot as plt
print("Generator:", GEN.name, "| model:", getattr(GEN, "model_id", "-"))
print("Bug DUY NHẤT:",RESULT["seen"],"| test:",RESULT["total"],
      "| hợp lệ: %.1f%%"%(100*RESULT["valid"]/max(1,RESULT["total"])),
      "| GPU runs:",RESULT["gpu_runs"],"| cov-cells:",RESULT["cov"])
print("stats:",RESULT["stats"])
df=pd.DataFrame(RESULT["crashes"])
if len(df):
    print("\nBảng bug (op | loại | tín hiệu):")
    print(df.groupby(["op","status","signal"]).size().to_string())
    print("\nReproducer đã tối giản (steps_full -> steps_min):")
    print(df[["op","status","steps_full","steps_min","src"]].to_string(index=False))
cc=np.array(RESULT["curve"],dtype=float)
fig,ax=plt.subplots(1,2,figsize=(11,4))
if len(cc):
    ax[0].plot(cc[:,0],cc[:,1],color="crimson"); ax[0].set_title("Bug duy nhất tích lũy")
    ax[0].set_xlabel("so test"); ax[0].set_ylabel("bug")
    vr=100*cc[:,2]/np.maximum(1,cc[:,0])
    ax[1].plot(cc[:,0],vr,color="teal"); ax[1].set_title("Tỉ lệ hợp lệ (%)")
    ax[1].set_xlabel("so test"); ax[1].set_ylabel("%"); ax[1].set_ylim(0,100)
plt.tight_layout(); plt.savefig("acg_report_12b.png",dpi=110); acg_copy_to_drive("acg_report_12b.png") if "acg_copy_to_drive" in globals() else None; plt.show()
with open("acg_crashes_12b.jsonl","w",encoding="utf-8") as f:
    for r in RESULT["crashes"]: f.write(json.dumps(r,ensure_ascii=False)+"\n")
acg_copy_to_drive("acg_crashes_12b.jsonl") if "acg_copy_to_drive" in globals() else None
if globals().get("ACG_DRIVE_DIR"):
    print(">> Da mirror bao cao sang Drive:", ACG_DRIVE_DIR)
print("\n>> Đã lưu: acg_crashes_12b.jsonl, acg_report_12b.png")
try:
    from IPython.display import display
    if len(df): display(df.head(12))
except Exception: pass

Generator: gemma | model: google/gemma-4-12B-it


NameError: name 'RESULT' is not defined

In [ ]:
# === 12. Kiểm tra lại bug tối giản để lọc stable/flaky ===
import json, time
from collections import Counter

src = "acg_crashes_12b.jsonl"
try:
    rows = [json.loads(x) for x in open(src, encoding="utf-8") if x.strip()]
except FileNotFoundError:
    src = "acg_crashes_12b.jsonl"
    rows = [json.loads(x) for x in open(src, encoding="utf-8") if x.strip()]

def recheck_one(row, repeats=5):
    prog = row["program"]
    got = []
    for _ in range(repeats):
        r = EX.run_devdiff(prog, CFG)
        got.append((r.get("status"), r.get("signal",""), str(r.get("detail",""))[:80]))
    return got

stable = []
flaky = []
for i, row in enumerate(rows):
    got = recheck_one(row, repeats=5)
    c = Counter(x[0] for x in got)
    row2 = dict(row)
    row2["recheck"] = got
    row2["recheck_status_counts"] = dict(c)
    if c.get(row["status"], 0) >= 4:
        stable.append(row2)
    else:
        flaky.append(row2)

print("total:", len(rows), "| stable:", len(stable), "| flaky:", len(flaky))
print("stable ops:", Counter((r["op"], r["status"]) for r in stable))
print("flaky ops:", Counter((r["op"], r["status"]) for r in flaky))

with open("acg_stable_bugs_12b.jsonl", "w", encoding="utf-8") as f:
    for r in stable:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open("acg_flaky_bugs_12b.jsonl", "w", encoding="utf-8") as f:
    for r in flaky:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

acg_copy_to_drive("acg_stable_bugs_12b.jsonl") if "acg_copy_to_drive" in globals() else None
acg_copy_to_drive("acg_flaky_bugs_12b.jsonl") if "acg_copy_to_drive" in globals() else None
print("ÄÃ£ lÆ°u: acg_stable_bugs_12b.jsonl, acg_flaky_bugs_12b.jsonl")
if globals().get("ACG_DRIVE_DIR"):
    print(">> Da mirror recheck sang Drive:", ACG_DRIVE_DIR)

In [ ]:
# === 13. Xuất toàn bộ artifact 12B thành tệp nén ===
import os, json, glob, time, shutil, subprocess, platform, textwrap
from pathlib import Path

EXPORT_DIR = (Path(ACG_DRIVE_DIR) / "acg_12b_export") if globals().get("ACG_DRIVE_DIR") else Path("/content/acg_12b_export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

stamp = time.strftime("%Y%m%d_%H%M%S")
zip_base = ((Path(ACG_DRIVE_DIR) / f"acg_12b_export_{stamp}") if globals().get("ACG_DRIVE_DIR") else Path(f"/content/acg_12b_export_{stamp}"))

def safe_json(obj):
    try:
        return json.loads(json.dumps(obj, ensure_ascii=False, default=str))
    except Exception:
        return str(obj)

# 1. Copy known output files
patterns = [
    "/content/acg_*.jsonl",
    "/content/acg_*.json",
    "/content/acg_*.png",
    "/content/*.jsonl",
]
for pat in patterns:
    for src in glob.glob(pat):
        try:
            shutil.copy2(src, EXPORT_DIR / Path(src).name)
        except Exception as e:
            print("skip copy", src, e)

# 2. Dump live Python variables
live = {}
for name in [
    "CFG", "OPS", "RESULT", "RESULT_HARD", "RESULT_E4B", "RESULT_12B",
    "GEN", "EX"
]:
    if name in globals():
        obj = globals()[name]
        if name == "CFG":
            live[name] = getattr(obj, "__dict__", str(obj))
        elif name == "GEN":
            live[name] = {
                "name": getattr(obj, "name", None),
                "model_id": getattr(obj, "model_id", None),
                "model_class": getattr(obj, "model_class", None),
            }
        elif name == "EX":
            live[name] = {
                "fw": getattr(obj, "fw", None),
            }
        else:
            live[name] = safe_json(obj)

(EXPORT_DIR / "live_state.json").write_text(
    json.dumps(live, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

# 3. Dump reproducers in one easy file
all_crashes = []
for key in ["RESULT", "RESULT_HARD", "RESULT_E4B", "RESULT_12B"]:
    r = globals().get(key)
    if isinstance(r, dict):
        for bug in r.get("crashes", []):
            b = dict(bug)
            b["_source_result"] = key
            all_crashes.append(b)

(EXPORT_DIR / "all_reproducers.json").write_text(
    json.dumps(all_crashes, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

# 4. Save worker source if available
if "WORKER_SRC" in globals():
    (EXPORT_DIR / "_acg_worker_12b.py").write_text(WORKER_SRC, encoding="utf-8")

# 5. Environment snapshot
env = {
    "time": stamp,
    "cwd": os.getcwd(),
    "python": platform.python_version(),
    "platform": platform.platform(),
}
try:
    env["nvidia_smi"] = subprocess.check_output(
        ["nvidia-smi"], text=True, stderr=subprocess.STDOUT
    )
except Exception as e:
    env["nvidia_smi"] = str(e)

try:
    env["pip_freeze"] = subprocess.check_output(
        ["pip", "freeze"], text=True, stderr=subprocess.STDOUT
    )
except Exception as e:
    env["pip_freeze"] = str(e)

(EXPORT_DIR / "environment.json").write_text(
    json.dumps(env, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8"
)

# 6. Short README
(EXPORT_DIR / "README.txt").write_text(
    textwrap.dedent(f"""
    ACG-LLM-Fuzz export
    Created: {stamp}

    Important files:
    - live_state.json: CFG + RESULT variables captured from RAM
    - all_reproducers.json: minimized bug/reproducer records
    - acg_crashes*.jsonl: raw crash logs saved by notebook
    - acg_report_12b.png: plot/report if generated
    - _acg_worker_12b.py: worker code used for execution
    - environment.json: GPU/package/runtime snapshot

    Colab runtime files originally lived under /content.
    """).strip() + "\n",
    encoding="utf-8"
)

# 7. Zip and download
zip_path = shutil.make_archive(str(zip_base), "zip", EXPORT_DIR)
print(">> Exported:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("Download manually from:", zip_path)
    print("download error:", e)